## MECANISMO PARA VER COMO EMPAQUETAR EL CÓDIGO PARA HACER MODELOS EN PRODUCCIÓN. 

In [5]:
# APROXIMACION INGENUA
# ----------------------------------------------------------------
# Empaquetado del entrenamiento del modelo
# entrena un modelo elastic net dando los parámetros
# y verbose es cuánta info va a mostrar


def train_estimator(alpha=0.5, l1_ratio=0.5, verbose=1):

    import os

    # para salvar(guardar) el modelo
    import pickle

    import pandas as pd

    # regresión elastic net robusto a outliers o puntos extremos
    # de los datos para evitar ese sobre-ajuste. Si no tengo
    # términos de penalización de outliers voy a tener
    # un modelo con problemas de ajuste
    from sklearn.linear_model import ElasticNet 

    # valores de error
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    # para separar la data
    from sklearn.model_selection import train_test_split

    INPUT_FILE = "../data//winequality-red.csv"
    OUTPUT_FOLDER = "../submission"
    ESTIMATOR_FILE = f"{OUTPUT_FOLDER}/estimator.pkl"

    # leer data
    df = pd.read_csv(INPUT_FILE)

    # Separo las predictorias de la objetivo
    y = df["quality"]
    x = df.copy()
    x.pop("quality")

    # partición de los datos
    (x_train, x_test, y_train, y_test) = train_test_split(
        x,
        y,
        test_size=0.25,
        random_state=0,
    )

    # instancia del modelo
    estimator = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=12345)

    # calculo de los estimadores del modelo
    estimator.fit(x_train, y_train)

    # predicciones del modelo
    y_pred = estimator.predict(x_test)

    # cálculo de métricas
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    if verbose > 0:
        print(estimator, ":", sep="")
        print(f"  MSE: {mse}")
        print(f"  MAE: {mae}")
        print(f"  R2: {r2}")


    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)

    if not os.path.exists(ESTIMATOR_FILE):
        saved_estimator = None
    else:
        with open(ESTIMATOR_FILE, "rb") as file:
            saved_estimator = pickle.load(file)

    # si no hay modelo en disco, lo guardamos
    # si existe modelo en disco lo compara
    # y guarda el mejor
    if saved_estimator is None or estimator.score(
        x_test, y_test
    ) > saved_estimator.score(x_test, y_test):
        with open(ESTIMATOR_FILE, "wb") as file:
            pickle.dump(estimator, file)

# y la idea es hacer diferentes intento de a1 y l1 para
# tantear que tan bueno es el modelo

In [6]:
# Experimento
train_estimator(0.5, 0.5)

ElasticNet(alpha=0.5, random_state=12345):
  MSE: 0.5294843132862007
  MAE: 0.5894666734018875
  R2: 0.13368827268570616


In [7]:
# Experimento
train_estimator(0.2, 0.2)

ElasticNet(alpha=0.2, l1_ratio=0.2, random_state=12345):
  MSE: 0.43869119518947153
  MAE: 0.5236106762028768
  R2: 0.2822387414965034


In [8]:
# Experimento
train_estimator(0.2, 0.2)

ElasticNet(alpha=0.2, l1_ratio=0.2, random_state=12345):
  MSE: 0.43869119518947153
  MAE: 0.5236106762028768
  R2: 0.2822387414965034


In [9]:
# Para usar el modelo
def use_estimator():

    import pandas as pd
    import pickle

    DATA_FILE = "../data/winequality-red.csv"
    OUTPUT_FOLDER = "../submission"
    ESTIMATOR_FILE = f"{OUTPUT_FOLDER}/estimator.pkl"

    df = pd.read_csv(DATA_FILE)

    y = df["quality"]
    x = df.copy()
    x.pop("quality")

    with open(ESTIMATOR_FILE, "rb") as file:
        estimator = pickle.load(file)

    y_pred = estimator.predict(x)

    return y_pred


use_estimator()


array([5.3398286 , 5.4181238 , 5.41836123, ..., 5.77802097, 5.5401108 ,
       5.71068002])

Principio DRY

D: DON'T
R: REPEAT
y: YOURSELF

y la idea es no repetir el código porque si hay erroress hay que cambiarlo en todos los lugares donde se repite el código. 

Entrando en el mundo profesional ya se tiene en cuenta este tipo de buenas prácticas. No es solo hacer modelos, este modelo tiene que ser capaz de entrar a productivo. Es necesario hacer código que pueda llevarse a productivo. **MLOPS** hay que llegar a **código de nivel profesional**. Entonces, factoricemos el código. Centrarnos en el algoritmo que hacemos a mano. 

In [10]:
# REFACTORIZACION DE CODIGO
# ----------------------------------------------------------------
# 1. Carga de datos


def load_data():

    import pandas as pd

    INPUT_FILE = "../data/winequality-red.csv"

    df = pd.read_csv(INPUT_FILE)

    y = df["quality"]
    x = df.copy()
    x.pop("quality")

    return x, y
  
  
  
  
  

In [11]:
# Particionamiento de datos


def make_train_test_split(x, y):

    from sklearn.model_selection import train_test_split

    (x_train, x_test, y_train, y_test) = train_test_split(
        x,
        y,
        #Garantizar que el modelo o los modelos 
        # siempre queden particionados de la misma manera en los
        # datos. 
        test_size=0.25,
        random_state=0,
    )
    return x_train, x_test, y_train, y_test


In [12]:
# Cálculo de metricas de evaluación


def eval_metrics(y_true, y_pred):

    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return mse, mae, r2  


In [13]:
# Reporte de métricas de evaluación


def report(estimator, mse, mae, r2):

    print(estimator, ":", sep="")
    print(f"  MSE: {mse}")
    print(f"  MAE: {mae}")
    print(f"  R2: {r2}")  


In [14]:
# Almacenamiento del modelo
# y sobre-escribe el modelo en disco y la idea 
# es siempre guardar el mejor modelo a disco. 

def save_best_estimator(estimator):

    import os
    import pickle

    OUTPUT_FOLDER = "../submission"
    ESTIMATOR_FILE = f"{OUTPUT_FOLDER}/estimator.pkl"

    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)

    with open(ESTIMATOR_FILE, "wb") as file:
        pickle.dump(estimator, file)  


In [15]:
# Carga del modelo


def load_best_estimator():

    import os
    import pickle

    OUTPUT_FOLDER = "../submission"
    ESTIMATOR_FILE = f"{OUTPUT_FOLDER}/estimator.pkl"

    if not os.path.exists(ESTIMATOR_FILE):
        return None
    with open(ESTIMATOR_FILE, "rb") as file:
        estimator = pickle.load(file)

    return estimator


In [ ]:
 # Entrenamiento
 # Notese que en este caso es mucho más claro
# cuando se revisa el códgio que es lo que se está haciendo

def train_estimator(alpha=0.5, l1_ratio=0.5, verbose=1):

    from sklearn.linear_model import ElasticNet

    x, y = load_data()
    x_train, x_test, y_train, y_test = make_train_test_split(x, y)
    estimator = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=12345)
    estimator.fit(x_train, y_train)
    mse, mae, r2 = eval_metrics(y_test, y_pred=estimator.predict(x_test))
    if verbose > 0:
        report(estimator, mse, mae, r2)

    best_estimator = load_best_estimator()
    if best_estimator is None or estimator.score(x_test, y_test) > best_estimator.score(
        x_test, y_test
    ):
        best_estimator = estimator
       

    save_best_estimator(best_estimator)

In [ ]:
train_estimator(0.5,0.5)

In [ ]:
train_estimator(0.2,0.2)

In [17]:
# Chequeo


def check_estimator():
    # cargar data
    x, y = load_data()

    # particionar data
    _, x_test, _, y_test = make_train_test_split(x, y)

    # cargar estimador
    estimator = load_best_estimator()

    # calcule metricas
    mse, mae, r2 = eval_metrics(y_test, y_pred=estimator.predict(x_test))

    # reporte métricas
    report(estimator, mse, mae, r2)


# Debe coincidir con el mejor modelo encontrado en la celdas anteriores
check_estimator()

# Y notese que de esta manera es mucho más fácil usar el código
# ya que simplemente es organizar estos particionamientos 
# y nos ahorramos mucho código, de tal manera que intentar
# diferentes tipos de modelos sean fáciles de probar. 
# además que el código permite siempre mantener el mejor modelo
# posible. 


ElasticNet(alpha=0.2, l1_ratio=0.2, random_state=12345):
  MSE: 0.43869119518947153
  MAE: 0.5236106762028768
  R2: 0.2822387414965034


In [18]:

# Búsqueda iterativa de hiperparámetros


def make_hyperparameters_search(alphas, l1_ratios):

    for alpha in alphas:
        for l1_ratio in l1_ratios:
            train_estimator(alpha=alpha, l1_ratio=l1_ratio, verbose=0)

# en cambio que si se trata hacer esto con la primera versión 
# de código sería muy pesado y sería una perdida de tiempo



In [ ]:
import numpy as np

alphas = np.linspace(0.0001, 0.5, 10)
l1_ratios = np.linspace(0.0001, 0.5, 10)
make_hyperparameters_search(alphas, l1_ratios)
check_estimator()         

# sin embargo lo mejor es aplicar cross validation 
# pues también los resultados de estos hiperparámetros
# es debido a las particiones que se tienene de los datos
# por tanto, al hacer cross validation podemos hallar
# los mejores hiperparámeros que mejor generalizan 

ElasticNet(alpha=0.0001, l1_ratio=0.0001, random_state=12345):
  MSE: 0.40021745821413146
  MAE: 0.4848004855172136
  R2: 0.34518725328239785


In [ ]:
# Búsqueda usando GridSearchCV
# que es la búsqueda en malla usando 
# cross-validation que sería el producto cruzado
# del conjunto de los valores de los hiperparámetros


def train_estimator(alphas, l1_ratios, n_splits=5, verbose=1):

    from sklearn.linear_model import ElasticNet
    from sklearn.model_selection import GridSearchCV

    x, y = load_data()
    x_train, x_test, y_train, y_test = make_train_test_split(x, y)

    # -------------------------------------------------------------------------
    # Búsqueda de parámetros con validación cruzada
    #
    estimator = GridSearchCV(
        estimator=ElasticNet(
            random_state=12345,
        ),
        param_grid={
            "alpha": alphas,
            "l1_ratio": l1_ratios,
        },
        cv=n_splits,
        refit=True,
        verbose=0,
        return_train_score=False,
    )
    # -------------------------------------------------------------------------

    estimator.fit(x_train, y_train)

    estimator = estimator.best_estimator_

    mse, mae, r2 = eval_metrics(y_test, y_pred=estimator.predict(x_test))
    if verbose > 0:
        report(estimator, mse, mae, r2)

    best_estimator = load_best_estimator()
    if best_estimator is None or estimator.score(x_test, y_test) > best_estimator.score(
        x_test, y_test
    ):
        best_estimator = estimator
        

    save_best_estimator(best_estimator)

# si yo quisiera hacer esto con cualquier modelo, 
# asi sea un modelo de regresión lineal
# solo debo cambiar el "estimator" y parámetros.

# El punto crucial, es que si hay que hacer algún
# cambio en los modelos, simplemente cambio los valores
# en la función y todo estará listo. 

# quedaría faltando simplemente la implementación 
# que hizo el profe en clase del ejepmlo de la red
# neuronal. 


In [21]:
import numpy as np

train_estimator(
    alphas=np.linspace(0.0001, 0.5, 10),
    l1_ratios=np.linspace(0.0001, 0.5, 10),
    n_splits=5,
    verbose=1,
)

ElasticNet(alpha=0.0001, l1_ratio=0.0001, random_state=12345):
  MSE: 0.40021745821413146
  MAE: 0.4848004855172136
  R2: 0.34518725328239785
